# Parts 6–7 — Baseline Models and Experimental Design

This notebook defines three baselines representing different retrieval strategies, documents the validation protocol before improvement, and compares quality with CPU latency.

**Notebook status:** 120 development questions only; locked test queries used = 0.


In [1]:
from collections import Counter
from pathlib import Path
import csv
import hashlib
import html
import json
import platform
import random
import statistics

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

def load_json(relative_path):
    return json.loads((ROOT / relative_path).read_text(encoding="utf-8"))

def print_table(rows, columns):
    if not rows:
        print("(no rows)")
        return
    widths = {
        column: max(len(str(column)), *(len(str(row.get(column, ""))) for row in rows))
        for column in columns
    }
    print(" | ".join(str(column).ljust(widths[column]) for column in columns))
    print("-+-".join("-" * widths[column] for column in columns))
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

def write_bar_svg(filename, values, title, *, maximum=None):
    values = list(values)
    width, left, right, row_height = 820, 245, 80, 34
    height = 76 + row_height * len(values)
    plot_width = width - left - right
    largest = maximum or max((float(value) for _, value in values), default=1.0) or 1.0
    elements = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="white"/>',
        f'<text x="{width / 2}" y="27" text-anchor="middle" font-family="Arial" font-size="18" font-weight="700">{html.escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(values):
        y = 52 + index * row_height
        bar_width = plot_width * float(value) / largest
        elements.extend([
            f'<text x="{left - 10}" y="{y + 17}" text-anchor="end" font-family="Arial" font-size="13">{html.escape(str(label))}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="20" rx="3" fill="#1c5b58"/>',
            f'<text x="{min(left + bar_width + 7, width - 58):.2f}" y="{y + 16}" font-family="Arial" font-size="12">{float(value):.4g}</text>',
        ])
    elements.append('</svg>')
    target = FIGURES / filename
    target.write_text("\n".join(elements) + "\n", encoding="utf-8")
    print(f"Saved visualization: {target.relative_to(ROOT)}")
    return target

print(f"Project: {ROOT.name} | Python: {platform.python_version()} | fixed seed: {SEED}")


Project: h | Python: 3.12.13 | fixed seed: 20250816


## Part 6 — Why these three baselines?

1. **Direct multilingual dense retrieval:** tests whether a pretrained shared embedding space can cross the Roman/Urdu script boundary without explicit conversion.
2. **Single deterministic transliteration + BM25:** tests a classical lexical strategy after one Roman-to-Urdu conversion. It is interpretable but exposes the brittleness of committing to one spelling.
3. **Standard raw-query BM25 + dense RRF hybrid:** tests whether lexical and semantic signals complement each other without the proposed controlled query bridge.

These are genuinely different modeling strategies—semantic vector retrieval, transformed lexical retrieval, and hybrid rank fusion—rather than three parameter variants of the same model.


## Part 7 — Experimental methodology fixed before improvement

**Split strategy:** 180 evidence-linked queries are frozen as 120 development and 60 test. Development is used for diagnosis and configuration; the test partition remains locked for one final evaluation after all decisions are frozen.

**Validation approach:** The project reports macro-averaged retrieval metrics on the fixed development split. Cross-validation is not used because the systems are pretrained retrieval pipelines rather than supervised estimators fitted to these 120 queries, and because corpus indexing/reranking is expensive. The limitation is that development estimates have high uncertainty.

**Hyperparameter strategy:** Passage size, top-k, query-view threshold, RRF constant/weights, reranking depth, and evidence thresholds are stored in versioned configuration. Changes are justified through development ablation and failure analysis; the locked test set is never used for selection.

**Metrics:** Recall@1/5/10 measure evidence coverage; MRR@10 rewards an early first relevant result; nDCG@10 measures discounted ranking quality. Mean and p95 latency measure interactive cost. Retrieval metrics are not final answer-accuracy percentages.

**Reproducibility:** Fixed seed 20250816, CPU device, exact package pins, immutable model revisions, checksum-verified passage/embedding artifacts, and stored query order.


In [2]:
config = (ROOT / "configs/default.yaml").read_text(encoding="utf-8")
audit = load_json("reports/tables/portability_audit.json")
embedding = load_json("artifacts/metadata/e5_small_150_30.json")
reranker = load_json("reports/tables/reranker_depth20.json")
reproducibility = [
    {"control": "random seed", "value": SEED},
    {"control": "device", "value": embedding["device"].upper()},
    {"control": "validation OS", "value": platform.platform()},
    {"control": "validation processor", "value": platform.processor() or platform.machine()},
    {"control": "Python", "value": audit["checks"]["python_supported"]["detail"]},
    {"control": "exact Python pins", "value": audit["checks"]["python_dependencies_pinned"]["detail"]},
    {"control": "dense model", "value": embedding["model"]},
    {"control": "dense revision", "value": embedding["revision"][:16] + "..."},
    {"control": "reranker", "value": reranker["model"]},
    {"control": "reranker revision", "value": reranker["revision"][:16] + "..."},
    {"control": "passage checksum", "value": audit["checks"]["passage_checksum"]["detail"][:16] + "..."},
    {"control": "embedding checksum", "value": audit["checks"]["embedding_checksum"]["detail"][:16] + "..."},
    {"control": "locked-test report violations", "value": 0},
]
print_table(reproducibility, ["control", "value"])
assert "seed: 20250816" in config and audit["status"] == "passed"


control                       | value                                               
------------------------------+-----------------------------------------------------
random seed                   | 20250816                                            
device                        | CPU                                                 
validation OS                 | Windows-11-10.0.26200-SP0                           
validation processor          | Intel64 Family 6 Model 142 Stepping 12, GenuineIntel
Python                        | 3.12.13                                             
exact Python pins             | 21 exact pins                                       
dense model                   | intfloat/multilingual-e5-small                      
dense revision                | d1d99a1efae67793...                                 
reranker                      | Alibaba-NLP/gte-multilingual-reranker-base          
reranker revision             | a6258e9d2b1a11aa...              

## Baseline results


In [3]:
report = load_json("reports/tables/baselines_development.json")
assert report["queries"] == 120 and report["test_queries_used"] == 0
rows = []
for name, values in report["systems"].items():
    rows.append({"system": name, "R@1": f'{values["recall_at_1"]:.4f}', "R@5": f'{values["recall_at_5"]:.4f}', "R@10": f'{values["recall_at_10"]:.4f}', "MRR@10": f'{values["mrr_at_10"]:.4f}', "nDCG@10": f'{values["ndcg_at_10"]:.4f}', "mean_ms": f'{values["mean_latency_ms"]:.1f}', "p95_ms": f'{values["p95_latency_ms"]:.1f}'})
print_table(rows, ["system", "R@1", "R@5", "R@10", "MRR@10", "nDCG@10", "mean_ms", "p95_ms"])
write_bar_svg("baseline_recall_at_10.svg", [(name, values["recall_at_10"]) for name, values in report["systems"].items()], "Baseline Recall@10", maximum=0.20)


system                      | R@1    | R@5    | R@10   | MRR@10 | nDCG@10 | mean_ms | p95_ms
----------------------------+--------+--------+--------+--------+---------+---------+-------
direct_dense                | 0.0500 | 0.0833 | 0.0917 | 0.0615 | 0.0686  | 51.8    | 61.7  
single_transliteration_bm25 | 0.0000 | 0.0167 | 0.0250 | 0.0074 | 0.0116  | 60.8    | 101.1 
standard_hybrid             | 0.0250 | 0.0667 | 0.0917 | 0.0456 | 0.0565  | 94.5    | 127.3 
Saved visualization: reports\figures\baseline_recall_at_10.svg


![Baseline Recall at 10](../reports/figures/baseline_recall_at_10.svg)


## Quality–latency trade-off


In [4]:
latency_rows = []
for name, values in report["systems"].items():
    latency_rows.append({"system": name, "mean_ms": values["mean_latency_ms"], "p95_ms": values["p95_latency_ms"], "R@10 per 100 ms": round(values["recall_at_10"] / values["mean_latency_ms"] * 100, 4)})
print_table(latency_rows, ["system", "mean_ms", "p95_ms", "R@10 per 100 ms"])
write_bar_svg("baseline_latency.svg", [(row["system"], row["mean_ms"]) for row in latency_rows], "Baseline mean query latency (ms)")


system                      | mean_ms | p95_ms  | R@10 per 100 ms
----------------------------+---------+---------+----------------
direct_dense                | 51.788  | 61.666  | 0.177          
single_transliteration_bm25 | 60.758  | 101.098 | 0.0411         
standard_hybrid             | 94.522  | 127.279 | 0.097          
Saved visualization: reports\figures\baseline_latency.svg


![Baseline latency](../reports/figures/baseline_latency.svg)


## Baseline conclusion

Direct dense and the standard hybrid tie at 0.0917 Recall@10, but direct dense has better MRR@10 and lower latency, so it is the strongest baseline under the primary quality/cost trade-off. Single transliteration + BM25 reaches only 0.0250 Recall@10, demonstrating that one script conversion is too brittle. The low absolute recall and named-entity failures justify a multi-view, script-aware improvement.
